In [1]:
!pip install -q "great-expectations==0.18.19"

In [3]:
import pandas as pd
import great_expectations as gx
from great_expectations.data_context import FileDataContext


In [ ]:
clean_data = "data_clean.csv"

df = pd.read_csv(clean_data)
df.head(10)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,...,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license,city,scrape_date,has_price,has_license,host_segment,listing_key
0,13913,Holiday London DB Room Let-on going,54730,Alina,Not Available,Islington,51.56861,-0.11270,Private room,70.0,...,2,331,10,No License,London,2026-03-29,True,False,Commercial / Multi-Listing,London-13913
1,15400,Bright Chelsea Apartment. Chelsea!,60302,Philippa,Not Available,Kensington and Chelsea,51.48780,-0.16813,Entire home/apt,149.0,...,1,199,1,No License,London,2026-03-29,True,False,Single Listing,London-15400
2,17402,Very Central Modern 3-Bed/2 Bath By Oxford St W1,67564,Liz,Not Available,Westminster,51.52195,-0.14094,Entire home/apt,411.0,...,2,80,0,No License,London,2026-03-29,True,False,Commercial / Multi-Listing,London-17402
3,24328,Battersea live/work artist house,41759,Joe,Not Available,Wandsworth,51.47072,-0.16266,Entire home/apt,0.0,...,1,294,1,No License,London,2026-03-29,False,False,Single Listing,London-24328
4,36274,Bright 1 bedroom apt off brick lane in Shoreditch,133271,Hendryks,Not Available,Tower Hamlets,51.52322,-0.06979,Entire home/apt,210.0,...,2,323,6,No License,London,2026-03-29,True,False,Commercial / Multi-Listing,London-36274
5,36299,Kew Gardens 3BR house in cul-de-sac,155938,Geert,Not Available,Richmond upon Thames,51.48145,-0.28107,Entire home/apt,280.0,...,1,324,6,No License,London,2026-03-29,True,False,Single Listing,London-36299
6,36660,You are GUARANTEED to love this,157884,Agri & Roger,Not Available,Haringey,51.58478,-0.16057,Private room,90.0,...,2,289,39,No License,London,2026-03-29,True,False,Commercial / Multi-Listing,London-36660
7,38605,SUNNY ROOM PRIVATE BATHROOM PLUS BREAKFAST,165579,Elisa & Dom,Not Available,Hammersmith and Fulham,51.50681,-0.23345,Private room,61.0,...,3,9,0,No License,London,2026-03-29,True,False,Commercial / Multi-Listing,London-38605
8,38610,Short Term Home,165579,Elisa & Dom,Not Available,Hammersmith and Fulham,51.50701,-0.23362,Entire home/apt,340.0,...,3,317,0,No License,London,2026-03-29,True,False,Commercial / Multi-Listing,London-38610
9,38995,SPACIOUS ROOM IN CONTEMPORARY STYLE FLAT,167281,Cesar,Not Available,Southwark,51.47860,-0.06114,Private room,49.0,...,1,172,16,No License,London,2026-03-29,True,False,Single Listing,London-38995


In [5]:
df.shape

(292802, 24)

In [6]:
context = FileDataContext.create(project_root_dir='./gx/')

In [7]:
datasource = context.sources.add_pandas(name="airbnb_clean_source")  # Create a pandas datasource because the data is already in a DataFrame.

data_asset = datasource.add_dataframe_asset(name="airbnb_clean_asset")  # Register the DataFrame as a GX data asset.

batch_request = data_asset.build_batch_request(dataframe=df)  # Build a batch request that points GX to the current DataFrame.

In [8]:
expectation_suite_name = "airbnb_clean_suite"  

context.add_or_update_expectation_suite(expectation_suite_name) 

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=expectation_suite_name,
) 

validator.head() 

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,...,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license,city,scrape_date,has_price,has_license,host_segment,listing_key
0,13913,Holiday London DB Room Let-on going,54730,Alina,Not Available,Islington,51.56861,-0.11270,Private room,70.0,...,2,331,10,No License,London,2026-03-29,True,False,Commercial / Multi-Listing,London-13913
1,15400,Bright Chelsea Apartment. Chelsea!,60302,Philippa,Not Available,Kensington and Chelsea,51.48780,-0.16813,Entire home/apt,149.0,...,1,199,1,No License,London,2026-03-29,True,False,Single Listing,London-15400
2,17402,Very Central Modern 3-Bed/2 Bath By Oxford St W1,67564,Liz,Not Available,Westminster,51.52195,-0.14094,Entire home/apt,411.0,...,2,80,0,No License,London,2026-03-29,True,False,Commercial / Multi-Listing,London-17402
3,24328,Battersea live/work artist house,41759,Joe,Not Available,Wandsworth,51.47072,-0.16266,Entire home/apt,0.0,...,1,294,1,No License,London,2026-03-29,False,False,Single Listing,London-24328
4,36274,Bright 1 bedroom apt off brick lane in Shoreditch,133271,Hendryks,Not Available,Tower Hamlets,51.52322,-0.06979,Entire home/apt,210.0,...,2,323,6,No License,London,2026-03-29,True,False,Commercial / Multi-Listing,London-36274


# Expectation 1: Unique Key

In [9]:
validator.expect_column_values_to_be_unique("listing_key") 

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 292802,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# Expectation 2: Availability Range

In [10]:
validator.expect_column_values_to_be_between(
    column="availability_365",
    min_value=0,
    max_value=365,
) 

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 292802,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# Expectation 3: Room Type Set

In [11]:
validator.expect_column_values_to_be_in_set(
    column="room_type",
    value_set=["Entire home/apt", "Private room", "Hotel room", "Shared room"],
) 

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 292802,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# Expectation 4: table row count

In [16]:
validator.expect_table_row_count_to_equal(292802) 

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": 292802
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# Expectation 5: Exact distinct City values

In [17]:
validator.expect_column_distinct_values_to_equal_set(
    column="city",
    value_set=["Amsterdam", "Bangkok", "Barcelona", "London", "Paris", "Rome", "Sydney"],
)  

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": [
      "Amsterdam",
      "Bangkok",
      "Barcelona",
      "London",
      "Paris",
      "Rome",
      "Sydney"
    ],
    "details": {
      "value_counts": [
        {
          "value": "Amsterdam",
          "count": 10480
        },
        {
          "value": "Bangkok",
          "count": 28806
        },
        {
          "value": "Barcelona",
          "count": 19410
        },
        {
          "value": "London",
          "count": 96871
        },
        {
          "value": "Paris",
          "count": 81853
        },
        {
          "value": "Rome",
          "count": 37652
        },
        {
          "value": "Sydney",
          "count": 17730
        }
      ]
    }
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# Expectation 6: listing key regex

In [18]:
validator.expect_column_values_to_match_regex(
    column="listing_key",
    regex=r"^[A-Za-z]+-\d+$",
)  

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 292802,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# Expectation 7: availability range(median version)

In [19]:
validator.expect_column_median_to_be_between(
    column="availability_365",
    min_value=0,
    max_value=365
)

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": 155.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

# Checkpoint

In [22]:
validator.save_expectation_suite(discard_failed_expectations=False) 

In [23]:
checkpoint = context.add_or_update_checkpoint(
    name="airbnb_clean_checkpoint",
    validator=validator,
)  

In [24]:
checkpoint_result = checkpoint.run()  

checkpoint_result  

Calculating Metrics:   0%|          | 0/52 [00:00<?, ?it/s]

{
  "run_id": {
    "run_name": null,
    "run_time": "2026-04-26T23:49:24.906543+07:00"
  },
  "run_results": {
    "ValidationResultIdentifier::airbnb_clean_suite/__none__/20260426T164924.906543Z/airbnb_clean_source-airbnb_clean_asset": {
      "validation_result": {
        "success": true,
        "results": [
          {
            "success": true,
            "expectation_config": {
              "expectation_type": "expect_column_values_to_be_unique",
              "kwargs": {
                "column": "listing_key",
                "batch_id": "airbnb_clean_source-airbnb_clean_asset"
              },
              "meta": {}
            },
            "result": {
              "element_count": 292802,
              "unexpected_count": 0,
              "unexpected_percent": 0.0,
              "partial_unexpected_list": [],
              "missing_count": 0,
              "missing_percent": 0.0,
              "unexpected_percent_total": 0.0,
              "unexpected_percent_nonm